# Entrega 2 proyecto aprendizaje de máquina

Pedro Pablo Sanín Trujillo (202221527)

Daniel Esteban Pardo Pardo (202311415)

## 1 - Carga de datos


In [3]:
# Importar la librerías del proyecto
import pandas as pd
import tensorflow as tf
from keras.layers import TextVectorization
from sklearn.model_selection import train_test_split

# [Agregado para la sección 4] TensorFlow solo reserva la memoria de GPU que va necesitando,
# para dejarle espacio a PyTorch (XLM-R). Sin GPU (por ejemplo, en Windows) no hace nada.
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

In [4]:
# A continuación, se cargan los conjuntos de datos de entrenamiento y evaluación

path_train = "./datos/train.csv"
path_eval = "./datos/eval.csv"


dataset_train_original = pd.read_csv(path_train)
dataset_eval_original = pd.read_csv(path_eval)

dataset_train = dataset_train_original.copy()
dataset_eval = dataset_eval_original.copy()

In [5]:
print(
    f"Número de filas y columnas del conjunto de entrenamiento: {dataset_train.shape}"
)

dataset_train.head(10)

Número de filas y columnas del conjunto de entrenamiento: (20466, 3)


,id,text,label
0,0,Lo pedí un martes por internet y llegó al terc...,negativo
1,1,Les cuento como me fue. Compre esta tablet a m...,positivo
2,2,Lo compré hace como dos meses en una tienda de...,neutral
3,3,Le routeur capte très bien dans toute la maiso...,negativo
4,4,"I got it a while back on sale online, it arriv...",neutral
5,5,Comprei esse teclado mês passado pra trabalhar...,positivo
6,6,Comprei há umas duas semanas pela internet e c...,negativo
7,7,"La compre a mediados de año por internet, tard...",positivo
8,8,Les dejo un comentario rápido. Lo pedí a media...,negativo
9,9,The color accuracy on this monitor is genuinel...,positivo


Vemos que el dataset contiene 20466 filas, cada una con un id, la reseña correspondiente y un label que indica que si la reseña es positiva o negativa. Revisando manualmente algunas entradas, confirmamos que los únicos idiomas presentes en las reseñas son español, inglés, portugués, francés y una mezcla de español e inglés. Esto es consistente con lo indicado en el enunciado del proyecto. Preliminarmente, no separaremos los modelos reseñas por idioma pues esto reduce sustancialmente el tamaño de cada conjunto de datos. Sin embargo, si vale la pena identificar el idioma para tener una división train-test balanceada.

Además nuestra métrica de entrenamiento será el accuracy, de acuerdo con la competencia. A continuación, intentaremos identificar el idioma de cada una de nuestras reseñas.


## 2 - Identificando el idioma de la reseña

Para esto, descargaremos las 50 palabras más comunes en cada uno de los idiomas. Por la ley de Zipf, si las listas de las 50 palabras más comunes son distintas, una de las palabras aparecerá en la lista y esto nos servirá para determinar el idioma de la reseña. Asimismo, las palabras más comunes en cada idioma son artículos y generalmente tienen el mismo significado.

Así, el criterio para determinar el idioma de una reseña será el siguiente:

1. El primer y segundo idioma candidato serán los primeros dos conjuntos de palabras que tengan más coincidencia con las palabras de la reseña.

2. Con las palabras que pertenecen al conjunto más común,


In [6]:
import sys
import unicodedata
from collections import Counter

# Tabla de traducción que reemplaza por un espacio todo carácter Unicode de puntuación
# (categorías 'P*'), lo que incluye ¿ ¡ « » “ ” … además de la puntuación ASCII.
puntua = {
    i: " "
    for i in range(sys.maxunicode + 1)
    if unicodedata.category(chr(i)).startswith("P")
}

# Palabras de la lista de inglés que también son muy comunes en español ("lo he usado"),
# por lo que no cuentan como evidencia de inglés dentro de una reseña en español.
AMBIGUAS_EN = {"he"}


def procesar_resena(text):
    """Pasa la reseña a minúsculas, elimina la puntuación y la separa en palabras."""
    text = text.lower()
    text = text.translate(puntua)
    return text.split()


def palabras_exclusivas(palabras_por_idioma):
    """Para cada idioma, las palabras comunes que no aparecen en la lista de ningún otro idioma."""
    return {
        idioma: palabras
        - set().union(
            *(otras for otro, otras in palabras_por_idioma.items() if otro != idioma)
        )
        for idioma, palabras in palabras_por_idioma.items()
    }


def identificar_idioma(resena, palabras_por_idioma, exclusivas):
    """Identifica el idioma de una reseña: 'es', 'en', 'fr', 'pt_br', 'es_en' (bilingüe) o None.

    1. El idioma principal es el que tiene más ocurrencias de sus palabras exclusivas. Si la reseña
       no tiene ninguna palabra exclusiva, se usan todas las palabras comunes (incluidas las compartidas).
    2. Si el idioma principal es español o inglés y la reseña contiene al menos una palabra exclusiva
       del otro idioma, la reseña se considera bilingüe ('es_en').
    """
    conteo = Counter(procesar_resena(resena))

    def puntajes(listas):
        return {
            idioma: sum(n for palabra, n in conteo.items() if palabra in palabras)
            for idioma, palabras in listas.items()
        }

    puntaje = puntajes(exclusivas)
    if max(puntaje.values()) == 0:
        puntaje = puntajes(palabras_por_idioma)
        if max(puntaje.values()) == 0:
            return None
    principal = max(puntaje, key=lambda k: puntaje[k])

    hay_es = any(palabra in exclusivas["es"] for palabra in conteo)
    hay_en = any(palabra in exclusivas["en"] - AMBIGUAS_EN for palabra in conteo)
    if principal in ("es", "en") and hay_es and hay_en:
        return "es_en"
    return principal

In [7]:
# Cargamos las 50 palabras más comunes por idioma y les aplicamos la misma limpieza que a las reseñas,
# para que entradas como "c'", "l'" o "'s" queden como "c", "l" y "s" y puedan coincidir con las reseñas.
path_palabras = "./datos/50_palabras_mas_comunes/50_palabras_mas_comunes.csv"

palabras_comunes = pd.read_csv(path_palabras)
palabras_comunes["palabra"] = palabras_comunes["palabra"].map(
    lambda p: procesar_resena(p)[0]
)

# Conjunto de palabras por idioma, y el subconjunto de palabras exclusivas de cada idioma
palabras_por_idioma = palabras_comunes.groupby("idioma")["palabra"].apply(set).to_dict()
exclusivas = palabras_exclusivas(palabras_por_idioma)

palabras_comunes.head()

,idioma,rank,palabra,frecuencia
0,en,1,you,28787591
1,en,2,i,27086011
2,en,3,the,22761659
3,en,4,to,17099834
4,en,5,a,14484562


In [8]:
print(
    identificar_idioma("Esta es una reseña de prueba", palabras_por_idioma, exclusivas)
)
print(
    identificar_idioma("This is a test review", palabras_por_idioma, exclusivas)
)
print(
    identificar_idioma("Ceci est une critique de test", palabras_por_idioma, exclusivas)
)
print(
    identificar_idioma("Esta é uma crítica de teste", palabras_por_idioma, exclusivas)
)
print(
    identificar_idioma(
        "Esta es una reseña de prueba en inglés: This is a test review",
        palabras_por_idioma,
        exclusivas,
    )
)
print(
    identificar_idioma(
        "Esta es una reseña de prueba en francés: Ceci est une critique de test",
        palabras_por_idioma,
        exclusivas,
    )
)

es
en
fr
pt_br
es_en
es


In [9]:
# Identificamos el idioma de cada reseña de entrenamiento y evaluación
dataset_train["idioma"] = dataset_train["text"].map(
    lambda t: identificar_idioma(t, palabras_por_idioma, exclusivas)
)
dataset_eval["idioma"] = dataset_eval["text"].map(
    lambda t: identificar_idioma(t, palabras_por_idioma, exclusivas)
)

dataset_train["idioma"].value_counts(dropna=False, normalize=True)

idioma
es       0.652497
en       0.114678
pt_br    0.114043
fr       0.102511
es_en    0.016271
Name: proportion, dtype: float64

In [10]:
dataset_train.head()

,id,text,label,idioma
0,0,Lo pedí un martes por internet y llegó al terc...,negativo,es
1,1,Les cuento como me fue. Compre esta tablet a m...,positivo,es
2,2,Lo compré hace como dos meses en una tienda de...,neutral,es
3,3,Le routeur capte très bien dans toute la maiso...,negativo,fr
4,4,"I got it a while back on sale online, it arriv...",neutral,en


Ahora, dividimos el conjunto de datos en prueba y entrenamiento estratificado para incluir una proporción representativa de cada idioma y de cada etiqueta. Para esto, usaremos `sklearn.train_test_split`.


In [11]:
# Codificamos las etiquetas de como enteros
etiquetas = sorted(
    dataset_train["label"].unique()
)  # ['negativo', 'neutral', 'positivo']
etiqueta_a_id = {e: i for i, e in enumerate(etiquetas)}


# División 90/10 estratificada por idioma y etiqueta a la vez: cada combinación
# (p. ej. "fr|neutral") conserva la misma proporción en entrenamiento y prueba.
estrato = dataset_train["idioma"].astype(str) + "|" + dataset_train["label"]

# train_test_split exige al menos 2 reseñas por estrato. Las combinaciones con una sola
# reseña (solo hay una reseña bilingüe es_en neutral) no se pueden dividir, así que van
# directamente a entrenamiento.
raros = estrato.map(estrato.value_counts()) < 2

train_df, test_df = train_test_split(
    dataset_train[~raros],
    test_size=0.1,
    random_state=42,
    stratify=estrato[~raros],
)
train_df = pd.concat([train_df, dataset_train[raros]])


def dataset_a_tf(df, shuffle):
    ds = tf.data.Dataset.from_tensor_slices(
        (df["text"].values, df["label"].map(etiqueta_a_id).values)
    )
    if shuffle:
        ds = ds.shuffle(len(df), seed=42)
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)


train_dataset = dataset_a_tf(train_df, shuffle=True)
test_dataset = dataset_a_tf(test_df, shuffle=False)

## 2 - Red neuronal

In [12]:
encoder_mlp = TextVectorization(output_mode="tf_idf")
encoder_mlp.adapt(train_df["text"])

In [13]:
model_mlp = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(), dtype=tf.string),
        encoder_mlp,
    
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(3),
    ]
)

model_mlp.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 19045)          │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │       609,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 609,571 (2.33 MB)

 Trainable params: 609,571 (2.33 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
model_mlp.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

Ahora para entrenar el modelo, usaremos ``tf.keras.callbacks.EarlyStopping``. Esto nos permite parar el entrenamiento cuando la pérdida de validación (`val_loss`) deja de disminuir, y restaurar los parámetros de la época con menor pérdida de validación. Usamos la pérdida y no el accuracy porque el accuracy de validación puede seguir subiendo levemente mientras el modelo ya se está sobreajustando (la pérdida sube porque el modelo se vuelve demasiado confiado en sus errores).

In [15]:
# En lugar de fijar el número de épocas, entrenamos hasta un máximo de 50 y detenemos el
# entrenamiento cuando la métrica `monitor` de validación deja de mejorar durante `patience`
# épocas seguidas. restore_best_weights=True deja el modelo con los pesos de la mejor época.
# Por defecto se monitorea la pérdida de validación ("val_loss"), que se minimiza; si se pasa
# un accuracy (p. ej. monitor="val_accuracy"), se maximiza.
def entrenar_modelo_con_parada_temprana(
    model, train_dataset, test_dataset, max_epochs=50, patience=3, monitor="val_loss"
):
    modo = "min" if "loss" in monitor else "max"
    parada_temprana = tf.keras.callbacks.EarlyStopping(
        monitor=monitor,
        mode=modo,
        patience=patience,
        restore_best_weights=True,
        verbose=1,  # informa en qué época se detuvo y de qué época restauró los pesos
    )

    history = model.fit(
        train_dataset,
        epochs=max_epochs,
        validation_data=test_dataset,
        callbacks=[parada_temprana],
    )

    # Guardamos en el historial la época cuyos pesos restauró la parada temprana (índice desde 0),
    # para que los resúmenes usen exactamente esa época y no la recalculen con otra métrica.
    history.mejor_epoca = parada_temprana.best_epoch
    print(
        f"Mejor época según {monitor}: {history.mejor_epoca + 1} "
        f"(val_loss = {history.history['val_loss'][history.mejor_epoca]:.4f}, "
        f"val_accuracy = {history.history['val_accuracy'][history.mejor_epoca]:.4f})"
    )

    return history

In [16]:
# [Modificado para la sección 4] guardamos el historial para la tabla comparativa
historial_mlp = entrenar_modelo_con_parada_temprana(model_mlp, train_dataset, test_dataset)

Epoch 1/50


c:\Users\Administrador\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


576/576 ━━━━━━━━━━━━━━━━━━━━ 11s 15ms/step - accuracy: 0.6455 - loss: 0.5844 - val_accuracy: 0.6717 - val_loss: 0.5235
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.7969 - loss: 0.4021 - val_accuracy: 0.6878 - val_loss: 0.5535
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.8797 - loss: 0.2646 - val_accuracy: 0.6825 - val_loss: 0.6517
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step - accuracy: 0.9303 - loss: 0.1668 - val_accuracy: 0.6883 - val_loss: 0.7347
Epoch 4: early stopping
Restoring model weights from the end of the best epoch: 1.
Mejor época según val_loss: 1 (val_loss = 0.5235, val_accuracy = 0.6717)


### 3.1 - Búsqueda de hiperparámetros 

## 3 - Red neuronal recurrente básica


Inicialmente, usaremos una red neuronal recurrente para clasificar las reseñas. Inicialmente, veremos el desempeño de una RNN simple, donde cada neurona tiene información sobre las operaciones que ha hecho en pasos anteriores. Las redes neuronales recurrentes funcionan bien al procesar lenguaje natural pues el lenguaje tiene una naturaleza secuencial.

Usaremos la implementación de RNNs dada por `tf.keras.layers.SimpleRNN`. Para que el texto pueda pasar exitosamente por una red neuronal, debemos vectorizarlo. Para esto, usaremos una capa inicial de la red neuronal dada por `tf.keras.layers.TextVectorization`. Este algoritmo de vectorización combina varias formas de vectorizar frases como _tf_idf_, _count_ (bag of words) y soporta ngramas.


In [17]:
# Para una red recurrente necesitamos una secuencia de índices de palabras (output_mode='int'),
# no un vector tf_idf. Cada reseña se rellena o recorta a 100 palabras (mediana: 57, máximo: 125).


# Tildes a quitar (unicodedata no sirve aquí porque Keras pasa tensores, no str,
# y TensorFlow no trae normalización Unicode sin el paquete tensorflow_text).
TILDES = {
    "[áàâäã]": "a",
    "[éèêë]": "e",
    "[íìîï]": "i",
    "[óòôöõ]": "o",
    "[úùûü]": "u",
    "ñ": "n",
    "ç": "c",
}


def standardize_text(text):
    text = tf.strings.lower(text, encoding="utf-8")
    text = tf.strings.regex_replace(text, "<br />", " ")
    for patron, letra in TILDES.items():
        text = tf.strings.regex_replace(text, patron, letra)
    text = tf.strings.regex_replace(text, "[^a-z']", " ")
    return text


def crear_encoder_rnn(train_df, n_gram=1):

    encoder_rnn = TextVectorization(
        output_mode="int",
        output_sequence_length=100,
        standardize=standardize_text,  ngrams=n_gram  # pyright: ignore[reportArgumentType]
    )
    encoder_rnn.adapt(train_df["text"])
    return encoder_rnn


encoder_rnn = crear_encoder_rnn(train_df)

texto_original = train_df["text"].iloc[0]
texto_vectorizado = encoder_rnn(tf.convert_to_tensor([texto_original]))

encoder_rnn = crear_encoder_rnn(train_df)

texto_original = train_df["text"][1]
texto_vectorizado = encoder_rnn(tf.convert_to_tensor([texto_original]))
print(f"Texto original: {texto_original}")
print(f"Texto vectorizado: {texto_vectorizado}")

Texto original: Les cuento como me fue. Compre esta tablet a mediados de año y llego en cuatro dias a mi casa en Guadalajara, sin ningun problema con el paquete. En mi opinion, la resolucion me resulto ser deficiente para ser honesto. La uso sobre todo por las tardes para revisar correos y ver videos mientras ceno. La pantalla tactil termino siendo muy buena en el uso real.
Texto vectorizado: [[  82  687   69   10  185   18   76 1860    6   70    3  104    9   13
     4   52   12    6   20   24    4  417   42  746  150   19    5   95
     4   20  435    2  530   10   71  107  998   14  107  437    2   23
   147   53   15   36  163   14  732 1686    9  496  938  356 7770    2
   270  570  195  219  172  481    4    5   23  378    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0]]


Ahora, entrenaremos una red neuronal con una capa de embedding, una capa SimpleRNN de 64 neuronas y una capa oculta densa de 32 neuronas con función de activación ReLU. Asimismo, entrenamos un embedding con una dimensión de salida de 64. 


In [18]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(), dtype=tf.string),
        encoder_rnn,
        tf.keras.layers.Embedding(
            input_dim=encoder_rnn.vocabulary_size(), output_dim=64, mask_zero=True
        ),
        tf.keras.layers.SimpleRNN(64),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(3),
    ]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_2            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 100, 64)        │     1,127,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,137,539 (4.34 MB)

 Trainable params: 1,137,539 (4.34 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=["accuracy"],
)

In [20]:
entrenar_modelo_con_parada_temprana(model, train_dataset, test_dataset)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 27s 40ms/step - accuracy: 0.6314 - loss: 0.6425 - val_accuracy: 0.7611 - val_loss: 0.4507
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 21s 37ms/step - accuracy: 0.8319 - loss: 0.3455 - val_accuracy: 0.7772 - val_loss: 0.4408
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 21s 36ms/step - accuracy: 0.9134 - loss: 0.2014 - val_accuracy: 0.7665 - val_loss: 0.5528
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 22s 39ms/step - accuracy: 0.9585 - loss: 0.1032 - val_accuracy: 0.7533 - val_loss: 0.6902
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - accuracy: 0.9743 - loss: 0.0702 - val_accuracy: 0.7377 - val_loss: 0.8008
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.4408, val_accuracy = 0.7772)


### 2.1 - Probando diferentes combinaciones de capas

A continuación, probaremos 5 combinaciones distintas de capas para determinar cual tiene mejor desempeño en validación.

Para no repetir código, definimos una función que arma la red con la misma estructura de la celda anterior
(entrada de texto → `encoder_rnn` → `Embedding` → **capas recurrentes** → `Dense(32, relu)` → `Dense(3)`)
y solo cambia el bloque de capas recurrentes. Cada modelo tiene su propia capa de embedding, de modo que
ninguno comparte pesos entrenables con otro; `encoder_rnn` sí se comparte, pero no tiene pesos entrenables.

In [21]:
def construir_modelo_rnn(capas_recurrentes, nombre):
    """Arma y compila una red recurrente cuyo bloque central son las capas recibidas."""
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(), dtype=tf.string),
            encoder_rnn,
            tf.keras.layers.Embedding(
                input_dim=encoder_rnn.vocabulary_size(), output_dim=64, mask_zero=True
            ),
            *capas_recurrentes,
            tf.keras.layers.Dense(32, activation="relu"),
            tf.keras.layers.Dense(3),
        ],
        name=nombre,
    )
    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=tf.keras.optimizers.Adam(),
        metrics=["accuracy"],
    )
    return model


# Guardamos el historial de entrenamiento de cada combinación para compararlas al final
historiales = {}

#### Modelo 1 - SimpleRNN bidireccional

La misma `SimpleRNN(64)` del modelo base, pero envuelta en `Bidirectional`: una copia lee la reseña de
izquierda a derecha y otra de derecha a izquierda, y sus salidas se concatenan. Así, la representación final
no depende solo de las últimas palabras de la reseña.

In [22]:
model_rnn_1 = construir_modelo_rnn(
    [tf.keras.layers.Bidirectional(tf.keras.layers.SimpleRNN(64))],
    nombre="simplernn_bidireccional",
)
model_rnn_1.summary()

historiales["1 - SimpleRNN bidireccional"] = entrenar_modelo_con_parada_temprana(
    model_rnn_1, train_dataset, test_dataset
)

Model: "simplernn_bidireccional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_2            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, 100, 64)        │     1,127,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,147,843 (4.38 MB)

 Trainable params: 1,147,843 (4.38 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 33s 44ms/step - accuracy: 0.5966 - loss: 0.6733 - val_accuracy: 0.7171 - val_loss: 0.5106
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 24s 41ms/step - accuracy: 0.8192 - loss: 0.3679 - val_accuracy: 0.7753 - val_loss: 0.4675
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 23s 40ms/step - accuracy: 0.9243 - loss: 0.1879 - val_accuracy: 0.7416 - val_loss: 0.6467
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 20s 35ms/step - accuracy: 0.9572 - loss: 0.1167 - val_accuracy: 0.7499 - val_loss: 0.6806
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 20s 35ms/step - accuracy: 0.9757 - loss: 0.0639 - val_accuracy: 0.7587 - val_loss: 0.7861
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.4675, val_accuracy = 0.7753)


#### Modelo 2 - Dos SimpleRNN apiladas

Dos capas `SimpleRNN` apiladas (64 y 32 neuronas). La primera usa `return_sequences=True` para entregarle a la
segunda la salida de cada paso de tiempo, no solo la del último.

In [23]:
model_rnn_2 = construir_modelo_rnn(
    [
        tf.keras.layers.SimpleRNN(64, return_sequences=True),
        tf.keras.layers.SimpleRNN(32),
    ],
    nombre="simplernn_apiladas",
)
model_rnn_2.summary()

historiales["2 - SimpleRNN apiladas"] = entrenar_modelo_con_parada_temprana(
    model_rnn_2, train_dataset, test_dataset
)

Model: "simplernn_apiladas"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_2            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, 100, 64)        │     1,127,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 100, 64)        │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (None, 32)             │         3,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,139,619 (4.35 MB)

 Trainable params: 1,139,619 (4.35 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 29s 45ms/step - accuracy: 0.5826 - loss: 0.7359 - val_accuracy: 0.7098 - val_loss: 0.5357
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 25s 43ms/step - accuracy: 0.8193 - loss: 0.3761 - val_accuracy: 0.7513 - val_loss: 0.4702
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 24s 42ms/step - accuracy: 0.9255 - loss: 0.1835 - val_accuracy: 0.7553 - val_loss: 0.5661
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 28s 48ms/step - accuracy: 0.9601 - loss: 0.1070 - val_accuracy: 0.7435 - val_loss: 0.8102
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 28s 49ms/step - accuracy: 0.9746 - loss: 0.0700 - val_accuracy: 0.7504 - val_loss: 0.8694
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.4702, val_accuracy = 0.7513)


#### Modelo 3 - LSTM

Reemplazamos la `SimpleRNN` por una `LSTM(64)`. Sus compuertas de entrada, olvido y salida le permiten conservar
información a lo largo de secuencias largas y mitigan el desvanecimiento del gradiente de la RNN simple.

In [24]:
model_rnn_3 = construir_modelo_rnn(
    [tf.keras.layers.LSTM(64)],
    nombre="lstm",
)
model_rnn_3.summary()

historiales["3 - LSTM"] = entrenar_modelo_con_parada_temprana(
    model_rnn_3, train_dataset, test_dataset
)

Model: "lstm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_2            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_3 (Embedding)         │ (None, 100, 64)        │     1,127,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,162,307 (4.43 MB)

 Trainable params: 1,162,307 (4.43 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 39s 62ms/step - accuracy: 0.6577 - loss: 0.5732 - val_accuracy: 0.7670 - val_loss: 0.4151
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 35s 60ms/step - accuracy: 0.8357 - loss: 0.3134 - val_accuracy: 0.8065 - val_loss: 0.3314
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 31s 55ms/step - accuracy: 0.8777 - loss: 0.2325 - val_accuracy: 0.8080 - val_loss: 0.3946
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 34s 59ms/step - accuracy: 0.9022 - loss: 0.1860 - val_accuracy: 0.8119 - val_loss: 0.3834
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 31s 53ms/step - accuracy: 0.9266 - loss: 0.1469 - val_accuracy: 0.8080 - val_loss: 0.4651
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.3314, val_accuracy = 0.8065)


#### Modelo 4 - GRU bidireccional con dropout

Una `GRU(64)` bidireccional (similar a la LSTM pero con menos parámetros) seguida de un `Dropout(0.5)` para
regularizar, ya que con alrededor de 18.000 reseñas de entrenamiento los modelos se sobreajustan con facilidad.

In [25]:
model_rnn_4 = construir_modelo_rnn(
    [
        tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64)),
        tf.keras.layers.Dropout(0.5),
    ],
    nombre="gru_bidireccional_dropout",
)
model_rnn_4.summary()

historiales["4 - GRU bidireccional + dropout"] = entrenar_modelo_con_parada_temprana(
    model_rnn_4, train_dataset, test_dataset
)

Model: "gru_bidireccional_dropout"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_2            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_4 (Embedding)         │ (None, 100, 64)        │     1,127,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        49,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,181,251 (4.51 MB)

 Trainable params: 1,181,251 (4.51 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 41s 60ms/step - accuracy: 0.6284 - loss: 0.6188 - val_accuracy: 0.7587 - val_loss: 0.4559
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 36s 63ms/step - accuracy: 0.8331 - loss: 0.3303 - val_accuracy: 0.8056 - val_loss: 0.3435
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 36s 62ms/step - accuracy: 0.8844 - loss: 0.2387 - val_accuracy: 0.8017 - val_loss: 0.3726
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 35s 61ms/step - accuracy: 0.9188 - loss: 0.1782 - val_accuracy: 0.8168 - val_loss: 0.4081
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 35s 61ms/step - accuracy: 0.9406 - loss: 0.1349 - val_accuracy: 0.8158 - val_loss: 0.5063
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.3435, val_accuracy = 0.8056)


#### Modelo 5 - Dos LSTM bidireccionales apiladas con dropout

La combinación más profunda: dos `LSTM` bidireccionales apiladas (64 y 32 neuronas), con `Dropout(0.3)` entre ellas
y `Dropout(0.5)` antes de las capas densas.

In [26]:
model_rnn_5 = construir_modelo_rnn(
    [
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
        tf.keras.layers.Dropout(0.5),
    ],
    nombre="lstm_bidireccionales_apiladas",
)
model_rnn_5.summary()

historiales["5 - LSTM bidireccionales apiladas"] = entrenar_modelo_con_parada_temprana(
    model_rnn_5, train_dataset, test_dataset
)

Model: "lstm_bidireccionales_apiladas"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_2            │ (None, 100)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_5 (Embedding)         │ (None, 100, 64)        │     1,127,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 100, 128)       │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,236,547 (4.72 MB)

 Trainable params: 1,236,547 (4.72 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 62s 92ms/step - accuracy: 0.6041 - loss: 0.6132 - val_accuracy: 0.6312 - val_loss: 0.5397
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 60s 104ms/step - accuracy: 0.6447 - loss: 0.5178 - val_accuracy: 0.6488 - val_loss: 0.5389
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 60s 105ms/step - accuracy: 0.7693 - loss: 0.4208 - val_accuracy: 0.7924 - val_loss: 0.3635
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 68s 118ms/step - accuracy: 0.8670 - loss: 0.2683 - val_accuracy: 0.8012 - val_loss: 0.3738
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 69s 120ms/step - accuracy: 0.9052 - loss: 0.1980 - val_accuracy: 0.8041 - val_loss: 0.4015
Epoch 6/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 68s 118ms/step - accuracy: 0.9282 - loss: 0.1495 - val_accuracy: 0.8061 - val_loss: 0.5263
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 3.
Mejor época según val_loss: 3 (val_loss = 0.3635, val_accuracy = 0.7924)


#### Creación de archivos submission



In [27]:
# Métricas de la época que restauró la parada temprana (la de menor val_loss)
resumen_rnn = pd.DataFrame(
    [
        {
            "modelo": nombre,
            "mejor_epoca": h.mejor_epoca + 1,
            "val_loss": h.history["val_loss"][h.mejor_epoca],
            "val_accuracy": h.history["val_accuracy"][h.mejor_epoca],
            "train_accuracy": h.history["accuracy"][h.mejor_epoca],
        }
        for nombre, h in historiales.items()
    ]
).sort_values("val_accuracy", ascending=False)

resumen_rnn

,modelo,mejor_epoca,val_loss,val_accuracy,train_accuracy
2,3 - LSTM,2,0.331401,0.806546,0.835659
3,4 - GRU bidireccional + dropout,2,0.343469,0.805569,0.833053
4,5 - LSTM bidireccionales apiladas,3,0.363533,0.792379,0.769260
0,1 - SimpleRNN bidireccional,2,0.467510,0.775281,0.819154
1,2 - SimpleRNN apiladas,2,0.470199,0.751343,0.819317


In [28]:
def crear_archivo_predicciones(
    model, df, archivo_salida="./datos/submission.csv", columnas=("text",)
):
    """Predice la etiqueta de cada reseña de `df` y guarda un CSV con el formato de
    ./datos/sample_submission.csv: columnas `id` y `answer` ('negativo', 'neutral' o 'positivo').

    `columnas` son las columnas de `df` que recibe el modelo: con una sola (por defecto "text")
    se le pasan los textos directamente; con varias, un dict {nombre_columna: textos}, que es lo
    que esperan los modelos de dos entradas de la sección 3.2."""
    if len(columnas) == 1:
        entradas = df[columnas[0]].values
    else:
        entradas = {c: df[c].values for c in columnas}
    textos = tf.data.Dataset.from_tensor_slices(entradas).batch(32)

    # El modelo devuelve logits (una fila de 3 valores por reseña): la clase predicha es la de
    # mayor logit, y la traducimos de vuelta a texto con la misma lista usada al codificarlas.
    logits = model.predict(textos)
    clases = logits.argmax(axis=1)

    submission = pd.DataFrame(
        {"id": df["id"].values, "answer": [etiquetas[c] for c in clases]}
    )
    submission.to_csv(archivo_salida, index=False)

    print(f"Archivo guardado en {archivo_salida} ({len(submission)} predicciones)")
    print(submission["answer"].value_counts(normalize=True))
    return submission

In [29]:
# Cambiar por el modelo con mejor val_accuracy
modelos_rnn = [model_rnn_1, model_rnn_2, model_rnn_3, model_rnn_4, model_rnn_5]
nombres_modelo = ["Simple RNN", "Simple RNNs apiladas","LSTM","GRU","LSTM apiladas"]
modelos_rnn = {"nombre": nombres_modelo, "modelo": modelos_rnn}

for nombre, modelo in zip(modelos_rnn["nombre"], modelos_rnn["modelo"]):
    submission = crear_archivo_predicciones(modelo, dataset_eval, f"./datos/submission_{nombre}.csv")
    submission.head()

212/212 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step
Archivo guardado en ./datos/submission_Simple RNN.csv (6781 predicciones)
answer
negativo    0.407904
positivo    0.399941
neutral     0.192155
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step
Archivo guardado en ./datos/submission_Simple RNNs apiladas.csv (6781 predicciones)
answer
negativo    0.416015
positivo    0.382539
neutral     0.201445
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/step
Archivo guardado en ./datos/submission_LSTM.csv (6781 predicciones)
answer
positivo    0.407315
negativo    0.404808
neutral     0.187878
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 7s 29ms/step
Archivo guardado en ./datos/submission_GRU.csv (6781 predicciones)
answer
positivo    0.419112
negativo    0.384752
neutral     0.196136
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 14s 54ms/step
Archivo guardado en ./datos/submission_LSTM apiladas.csv (6781 predicciones)
answer


### 3.2 - Ajustando y regularizando los modelos RNN

En el proyecto anterior, la última oración era la que más significado daba a la reseña completa. Para esto, agregabamos la última oración como una entrada adicional al modelo. En este caso, haremos lo mismo.

Asimismo, vemos que los modelos que estamos usando tienden al overfitting. Para solucionar esto intentaremos pararemos el entrenamiento en las epochs en función de la función de pérdida y no el accuracy en el entrenamiento. De esta forma, el modelo elegido no se elige de acuerdo a los datos específicos de entrenamiento. 

También, intentaremos regularizar más las RNNs. Para esto, usaremos ``SpatialDropout1D`` después del embedding. A diferencia de ``Dropout``, ``SpatialDropout1D`` hace dropout para columnas enteras del espacio, lo que aumenta la regularización.



In [30]:
import re

def ultima_oracion(texto):
    """Devuelve la última oración de la reseña (corta después de . ! ?)."""
    oraciones = [o for o in re.split(r"(?<=[.!?])\s+", texto.strip()) if o]
    return oraciones[-1] if oraciones else texto

# .assign crea un DataFrame nuevo y evita el SettingWithCopyWarning
train_df = train_df.assign(ultima=train_df["text"].map(ultima_oracion))
test_df = test_df.assign(ultima=test_df["text"].map(ultima_oracion))
dataset_eval = dataset_eval.assign(ultima=dataset_eval["text"].map(ultima_oracion))


def entradas_modelo(df, columnas):
    """Una columna -> arreglo de textos (como antes); varias -> dict {nombre_columna: textos}."""
    if len(columnas) == 1:
        return df[columnas[0]].values
    return {c: df[c].values for c in columnas}


def dataset_a_tf(df, shuffle, columnas=("text",)):
    ds = tf.data.Dataset.from_tensor_slices(
        (entradas_modelo(df, columnas), df["label"].map(etiqueta_a_id).values)
    )
    if shuffle:
        ds = ds.shuffle(len(df), seed=42)
    return ds.batch(32).prefetch(tf.data.AUTOTUNE)


train_dataset = dataset_a_tf(train_df, shuffle=True)
test_dataset = dataset_a_tf(test_df, shuffle=False)



Para comparar de forma justa, cada modelo mejorado conserva exactamente el bloque recurrente de su
modelo original de la sección 2.1. Solo cambian dos cosas:

1. Una segunda entrada con la última oración de la reseña, procesada por su propia copia del bloque recurrente.
2. `SpatialDropout1D(0.3)` después del embedding.

La parada temprana según `val_loss` ya no es una diferencia entre ambos grupos: `entrenar_modelo_con_parada_temprana`
la usa por defecto, así que los modelos originales también se eligen con la pérdida de validación.

In [31]:
def construir_modelo_rnn_mejorado(
    crear_capas_recurrentes, nombre, tasa_spatial_dropout=0.3
):
    """Igual que `construir_modelo_rnn`, pero con las mejoras de esta sección:

    - Dos entradas: la reseña completa (`text`) y su última oración (`ultima`). Cada una pasa
      por su propia copia del bloque recurrente y las dos salidas se concatenan antes de las
      capas densas.
    - `SpatialDropout1D` justo después del embedding, en ambas ramas.

    `crear_capas_recurrentes` es una función que devuelve una lista NUEVA de capas cada vez que se
    llama, para que cada rama tenga sus propios pesos (si se usaran las mismas capas en las dos
    ramas, Keras compartiría sus pesos). El embedding sí se comparte entre ambas ramas, de modo
    que las palabras de la última oración también aprenden de la reseña completa.
    """
    embedding = tf.keras.layers.Embedding(
        input_dim=encoder_rnn.vocabulary_size(), output_dim=64, mask_zero=True
    )

    def rama(entrada):
        x = embedding(encoder_rnn(entrada))
        x = tf.keras.layers.SpatialDropout1D(tasa_spatial_dropout)(x)
        for capa in crear_capas_recurrentes():
            x = capa(x)
        return x

    # Los nombres de los Input coinciden con las columnas del DataFrame (las claves del dict)
    texto = tf.keras.layers.Input(shape=(), dtype=tf.string, name="text")
    ultima = tf.keras.layers.Input(shape=(), dtype=tf.string, name="ultima")

    x = tf.keras.layers.Concatenate()([rama(texto), rama(ultima)])
    x = tf.keras.layers.Dense(32, activation="relu")(x)
    salida = tf.keras.layers.Dense(3)(x)

    model = tf.keras.Model([texto, ultima], salida, name=nombre)
    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=tf.keras.optimizers.Adam(),
        metrics=["accuracy"],
    )
    return model


# Conjuntos de datos con las dos entradas: reseña completa y última oración
columnas_2 = ("text", "ultima")
train_dataset_2 = dataset_a_tf(train_df, shuffle=True, columnas=columnas_2)
test_dataset_2 = dataset_a_tf(test_df, shuffle=False, columnas=columnas_2)

# Usamos las mismas claves que en `historiales` para poder comparar cada modelo con su original
historiales_mejorados = {}

#### Modelo 1 mejorado - SimpleRNN bidireccional

Mismo bloque recurrente que el modelo 1: una `SimpleRNN(64)` bidireccional.

In [32]:
model_rnn_1_mejorado = construir_modelo_rnn_mejorado(
    lambda: [tf.keras.layers.Bidirectional(tf.keras.layers.SimpleRNN(64))],
    nombre="simplernn_bidireccional_mejorado",
)
model_rnn_1_mejorado.summary()

historiales_mejorados["1 - SimpleRNN bidireccional"] = entrenar_modelo_con_parada_temprana(
    model_rnn_1_mejorado, train_dataset_2, test_dataset_2, monitor="val_loss"
)

Model: "simplernn_bidireccional_mejorado"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ultima (InputLayer) │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 100)       │          0 │ text[0][0],       │
│ (TextVectorization) │                   │            │ ultima[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_6         │ (None, 100, 64)   │  1,127,104 │ text_vectorizati… │
│ (Embedding)         │                   │            │ text_vectorizati… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d   │ (None, 100, 64)   │          0 │ embedding_6[0][0] │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_6         │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_1 │ (None, 100, 64)   │          0 │ embedding_6[1][0] │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_7         │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_4     │ (None, 128)       │     16,512 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │ not_equal_6[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_5     │ (None, 128)       │     16,512 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │ not_equal_7[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 256)       │          0 │ bidirectional_4[… │
│ (Concatenate)       │                   │            │ bidirectional_5[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 32)        │      8,224 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 3)         │         99 │ dense_14[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,168,451 (4.46 MB)

 Trainable params: 1,168,451 (4.46 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50


c:\Users\Administrador\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


576/576 ━━━━━━━━━━━━━━━━━━━━ 67s 94ms/step - accuracy: 0.6421 - loss: 0.6278 - val_accuracy: 0.7714 - val_loss: 0.4489
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 49s 85ms/step - accuracy: 0.8282 - loss: 0.3532 - val_accuracy: 0.7753 - val_loss: 0.4082
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 50s 87ms/step - accuracy: 0.8902 - loss: 0.2414 - val_accuracy: 0.7758 - val_loss: 0.4449
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 48s 83ms/step - accuracy: 0.9238 - loss: 0.1833 - val_accuracy: 0.7807 - val_loss: 0.5485
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 52s 91ms/step - accuracy: 0.9501 - loss: 0.1221 - val_accuracy: 0.7709 - val_loss: 0.6136
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.4082, val_accuracy = 0.7753)


#### Modelo 2 mejorado - Dos SimpleRNN apiladas

Mismo bloque recurrente que el modelo 2: `SimpleRNN(64, return_sequences=True)` seguida de `SimpleRNN(32)`.

In [33]:
model_rnn_2_mejorado = construir_modelo_rnn_mejorado(
    lambda: [
        tf.keras.layers.SimpleRNN(64, return_sequences=True),
        tf.keras.layers.SimpleRNN(32),
    ],
    nombre="simplernn_apiladas_mejorado",
)
model_rnn_2_mejorado.summary()

historiales_mejorados["2 - SimpleRNN apiladas"] = entrenar_modelo_con_parada_temprana(
    model_rnn_2_mejorado, train_dataset_2, test_dataset_2, monitor="val_loss"
)

Model: "simplernn_apiladas_mejorado"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ultima (InputLayer) │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 100)       │          0 │ text[0][0],       │
│ (TextVectorization) │                   │            │ ultima[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_7         │ (None, 100, 64)   │  1,127,104 │ text_vectorizati… │
│ (Embedding)         │                   │            │ text_vectorizati… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_2 │ (None, 100, 64)   │          0 │ embedding_7[0][0] │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_8         │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_3 │ (None, 100, 64)   │          0 │ embedding_7[1][0] │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_9         │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_6        │ (None, 100, 64)   │      8,256 │ spatial_dropout1… │
│ (SimpleRNN)         │                   │            │ not_equal_8[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_8        │ (None, 100, 64)   │      8,256 │ spatial_dropout1… │
│ (SimpleRNN)         │                   │            │ not_equal_9[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_7        │ (None, 32)        │      3,104 │ simple_rnn_6[0][… │
│ (SimpleRNN)         │                   │            │ not_equal_8[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_9        │ (None, 32)        │      3,104 │ simple_rnn_8[0][… │
│ (SimpleRNN)         │                   │            │ not_equal_9[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 64)        │          0 │ simple_rnn_7[0][… │
│ (Concatenate)       │                   │            │ simple_rnn_9[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_16 (Dense)    │ (None, 32)        │      2,080 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_17 (Dense)    │ (None, 3)         │         99 │ dense_16[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,152,003 (4.39 MB)

 Trainable params: 1,152,003 (4.39 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 48s 66ms/step - accuracy: 0.6249 - loss: 0.7123 - val_accuracy: 0.7474 - val_loss: 0.5292
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 37s 63ms/step - accuracy: 0.8102 - loss: 0.4298 - val_accuracy: 0.7513 - val_loss: 0.5185
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 39s 67ms/step - accuracy: 0.8751 - loss: 0.3016 - val_accuracy: 0.7572 - val_loss: 0.5591
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 41s 71ms/step - accuracy: 0.9118 - loss: 0.2211 - val_accuracy: 0.7386 - val_loss: 0.6611
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 38s 67ms/step - accuracy: 0.9358 - loss: 0.1667 - val_accuracy: 0.7313 - val_loss: 0.8157
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.5185, val_accuracy = 0.7513)


#### Modelo 3 mejorado - LSTM

Mismo bloque recurrente que el modelo 3: una `LSTM(64)`.

In [34]:
model_rnn_3_mejorado = construir_modelo_rnn_mejorado(
    lambda: [tf.keras.layers.LSTM(64)],
    nombre="lstm_mejorado",
)
model_rnn_3_mejorado.summary()

historiales_mejorados["3 - LSTM"] = entrenar_modelo_con_parada_temprana(
    model_rnn_3_mejorado, train_dataset_2, test_dataset_2, monitor="val_loss"
)

Model: "lstm_mejorado"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ultima (InputLayer) │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 100)       │          0 │ text[0][0],       │
│ (TextVectorization) │                   │            │ ultima[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_8         │ (None, 100, 64)   │  1,127,104 │ text_vectorizati… │
│ (Embedding)         │                   │            │ text_vectorizati… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_4 │ (None, 100, 64)   │          0 │ embedding_8[0][0] │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_10        │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_5 │ (None, 100, 64)   │          0 │ embedding_8[1][0] │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_11        │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ (None, 64)        │     33,024 │ spatial_dropout1… │
│                     │                   │            │ not_equal_10[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_4 (LSTM)       │ (None, 64)        │     33,024 │ spatial_dropout1… │
│                     │                   │            │ not_equal_11[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 128)       │          0 │ lstm_3[0][0],     │
│ (Concatenate)       │                   │            │ lstm_4[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 32)        │      4,128 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 3)         │         99 │ dense_18[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,197,379 (4.57 MB)

 Trainable params: 1,197,379 (4.57 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 45s 66ms/step - accuracy: 0.6757 - loss: 0.5477 - val_accuracy: 0.7851 - val_loss: 0.3673
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 42s 73ms/step - accuracy: 0.8387 - loss: 0.3079 - val_accuracy: 0.8070 - val_loss: 0.3480
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 35s 61ms/step - accuracy: 0.8693 - loss: 0.2520 - val_accuracy: 0.7978 - val_loss: 0.3989
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 26s 45ms/step - accuracy: 0.8921 - loss: 0.2134 - val_accuracy: 0.7953 - val_loss: 0.4217
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 24s 42ms/step - accuracy: 0.9087 - loss: 0.1778 - val_accuracy: 0.7992 - val_loss: 0.5008
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.3480, val_accuracy = 0.8070)


#### Modelo 4 mejorado - GRU bidireccional con dropout

Mismo bloque recurrente que el modelo 4: una `GRU(64)` bidireccional seguida de `Dropout(0.5)`.

In [35]:
model_rnn_4_mejorado = construir_modelo_rnn_mejorado(
    lambda: [
        tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64)),
        tf.keras.layers.Dropout(0.5),
    ],
    nombre="gru_bidireccional_dropout_mejorado",
)
model_rnn_4_mejorado.summary()

historiales_mejorados["4 - GRU bidireccional + dropout"] = entrenar_modelo_con_parada_temprana(
    model_rnn_4_mejorado, train_dataset_2, test_dataset_2, monitor="val_loss"
)

Model: "gru_bidireccional_dropout_mejorado"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ultima (InputLayer) │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 100)       │          0 │ text[0][0],       │
│ (TextVectorization) │                   │            │ ultima[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_9         │ (None, 100, 64)   │  1,127,104 │ text_vectorizati… │
│ (Embedding)         │                   │            │ text_vectorizati… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_6 │ (None, 100, 64)   │          0 │ embedding_9[0][0] │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_12        │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_7 │ (None, 100, 64)   │          0 │ embedding_9[1][0] │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_13        │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_6     │ (None, 128)       │     49,920 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │ not_equal_12[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_7     │ (None, 128)       │     49,920 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │ not_equal_13[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ bidirectional_6[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 128)       │          0 │ bidirectional_7[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 256)       │          0 │ dropout_3[0][0],  │
│ (Concatenate)       │                   │            │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 32)        │      8,224 │ concatenate_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 3)         │         99 │ dense_20[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,235,267 (4.71 MB)

 Trainable params: 1,235,267 (4.71 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 52s 79ms/step - accuracy: 0.6843 - loss: 0.5631 - val_accuracy: 0.7973 - val_loss: 0.3527
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 52s 89ms/step - accuracy: 0.8415 - loss: 0.3013 - val_accuracy: 0.8031 - val_loss: 0.3425
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 75s 130ms/step - accuracy: 0.8692 - loss: 0.2471 - val_accuracy: 0.8007 - val_loss: 0.4058
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 88s 153ms/step - accuracy: 0.8927 - loss: 0.2120 - val_accuracy: 0.8021 - val_loss: 0.4183
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 87s 151ms/step - accuracy: 0.9102 - loss: 0.1762 - val_accuracy: 0.8100 - val_loss: 0.4580
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.3425, val_accuracy = 0.8031)


#### Modelo 5 mejorado - Dos LSTM bidireccionales apiladas con dropout

Mismo bloque recurrente que el modelo 5: dos `LSTM` bidireccionales apiladas (64 y 32 neuronas) con `Dropout(0.3)` entre ellas y `Dropout(0.5)` al final.

In [36]:
model_rnn_5_mejorado = construir_modelo_rnn_mejorado(
    lambda: [
        tf.keras.layers.Bidirectional(
            tf.keras.layers.LSTM(64, return_sequences=True)
        ),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
        tf.keras.layers.Dropout(0.5),
    ],
    nombre="lstm_bidireccionales_apiladas_mejorado",
)
model_rnn_5_mejorado.summary()

historiales_mejorados["5 - LSTM bidireccionales apiladas"] = entrenar_modelo_con_parada_temprana(
    model_rnn_5_mejorado, train_dataset_2, test_dataset_2, monitor="val_loss"
)

Model: "lstm_bidireccionales_apiladas_mejorado"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text (InputLayer)   │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ultima (InputLayer) │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 100)       │          0 │ text[0][0],       │
│ (TextVectorization) │                   │            │ ultima[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_10        │ (None, 100, 64)   │  1,127,104 │ text_vectorizati… │
│ (Embedding)         │                   │            │ text_vectorizati… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_8 │ (None, 100, 64)   │          0 │ embedding_10[0][… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_14        │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_9 │ (None, 100, 64)   │          0 │ embedding_10[1][… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_15        │ (None, 100)       │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_8     │ (None, 100, 128)  │     66,048 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │ not_equal_14[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_10    │ (None, 100, 128)  │     66,048 │ spatial_dropout1… │
│ (Bidirectional)     │                   │            │ not_equal_15[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 100, 128)  │          0 │ bidirectional_8[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 100, 128)  │          0 │ bidirectional_10… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_9     │ (None, 64)        │     41,216 │ dropout_5[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal_14[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_11    │ (None, 64)        │     41,216 │ dropout_7[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal_15[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 64)        │          0 │ bidirectional_9[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 64)        │          0 │ bidirectional_11… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 128)       │          0 │ dropout_6[0][0],  │
│ (Concatenate)       │                   │            │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_22 (Dense)    │ (None, 32)        │      4,128 │ concatenate_4[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_23 (Dense)    │ (None, 3)         │         99 │ dense_22[0][0]    │
└─────────────────────┴───────────────────┴────────────┴─────────────────

 Total params: 1,345,859 (5.13 MB)

 Trainable params: 1,345,859 (5.13 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 121s 177ms/step - accuracy: 0.6500 - loss: 0.5866 - val_accuracy: 0.7792 - val_loss: 0.3958
Epoch 2/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 88s 154ms/step - accuracy: 0.8236 - loss: 0.3313 - val_accuracy: 0.7963 - val_loss: 0.3486
Epoch 3/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 92s 160ms/step - accuracy: 0.8640 - loss: 0.2652 - val_accuracy: 0.7958 - val_loss: 0.3702
Epoch 4/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 99s 173ms/step - accuracy: 0.8790 - loss: 0.2329 - val_accuracy: 0.7968 - val_loss: 0.4237
Epoch 5/50
576/576 ━━━━━━━━━━━━━━━━━━━━ 91s 158ms/step - accuracy: 0.8922 - loss: 0.2026 - val_accuracy: 0.8026 - val_loss: 0.5104
Epoch 5: early stopping
Restoring model weights from the end of the best epoch: 2.
Mejor época según val_loss: 2 (val_loss = 0.3486, val_accuracy = 0.7963)


#### Comparación con los modelos originales

Para cada modelo tomamos las métricas de la época que restauró la parada temprana. Tanto en los originales como
en los mejorados esa época se eligió con `val_loss`, así que el `val_accuracy` reportado no está sesgado por la
selección y ambos grupos se comparan en igualdad de condiciones. Con ~2.000 reseñas de validación, el error estándar
del accuracy es de casi 1 punto, así que diferencias menores a eso no son concluyentes.

In [37]:
def resumir_historiales(historiales):
    """Una fila por modelo con las métricas de la época restaurada por la parada temprana."""
    filas = []
    for nombre, h in historiales.items():
        mejor = h.mejor_epoca
        filas.append(
            {
                "modelo": nombre,
                "mejor_epoca": mejor + 1,
                "val_accuracy": h.history["val_accuracy"][mejor],
                "val_loss": h.history["val_loss"][mejor],
                "train_accuracy": h.history["accuracy"][mejor],
            }
        )
    return pd.DataFrame(filas)


comparacion = resumir_historiales(historiales).merge(
    resumir_historiales(historiales_mejorados),
    on="modelo",
    suffixes=("_original", "_mejorado"),
)
comparacion["diferencia_val_accuracy"] = (
    comparacion["val_accuracy_mejorado"] - comparacion["val_accuracy_original"]
)

comparacion[
    [
        "modelo",
        "mejor_epoca_original",
        "mejor_epoca_mejorado",
        "val_loss_original",
        "val_loss_mejorado",
        "val_accuracy_original",
        "val_accuracy_mejorado",
        "diferencia_val_accuracy",
        "train_accuracy_original",
        "train_accuracy_mejorado",
    ]
].sort_values("val_accuracy_mejorado", ascending=False)

,modelo,mejor_epoca_original,mejor_epoca_mejorado,val_loss_original,val_loss_mejorado,val_accuracy_original,val_accuracy_mejorado,diferencia_val_accuracy,train_accuracy_original,train_accuracy_mejorado
2,3 - LSTM,2,2,0.331401,0.347971,0.806546,0.807035,0.000489,0.835659,0.838699
3,4 - GRU bidireccional + dropout,2,2,0.343469,0.342546,0.805569,0.803127,-0.002443,0.833053,0.841522
4,5 - LSTM bidireccionales apiladas,3,2,0.363533,0.348638,0.792379,0.796287,0.003908,0.769260,0.823606
0,1 - SimpleRNN bidireccional,2,2,0.467510,0.408244,0.775281,0.775281,0.000000,0.819154,0.828221
1,2 - SimpleRNN apiladas,2,2,0.470199,0.518475,0.751343,0.751343,0.000000,0.819317,0.810196


In [38]:
# Archivos submission de los modelos mejorados (reciben la reseña y su última oración)
modelos_mejorados = {
    "Simple RNN": model_rnn_1_mejorado,
    "Simple RNNs apiladas": model_rnn_2_mejorado,
    "LSTM": model_rnn_3_mejorado,
    "GRU": model_rnn_4_mejorado,
    "LSTM apiladas": model_rnn_5_mejorado,
}

for nombre, modelo in modelos_mejorados.items():
    crear_archivo_predicciones(
        modelo,
        dataset_eval,
        f"./datos/submission_{nombre}_mejorado.csv",
        columnas=columnas_2,
    )

212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step
Archivo guardado en ./datos/submission_Simple RNN_mejorado.csv (6781 predicciones)
answer
positivo    0.451998
negativo    0.353340
neutral     0.194662
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step
Archivo guardado en ./datos/submission_Simple RNNs apiladas_mejorado.csv (6781 predicciones)
answer
negativo    0.439758
positivo    0.341690
neutral     0.218552
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step
Archivo guardado en ./datos/submission_LSTM_mejorado.csv (6781 predicciones)
answer
negativo    0.459814
positivo    0.346851
neutral     0.193334
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step
Archivo guardado en ./datos/submission_GRU_mejorado.csv (6781 predicciones)
answer
negativo    0.466598
positivo    0.343312
neutral     0.190090
Name: proportion, dtype: float64
212/212 ━━━━━━━━━━━━━━━━━━━━ 10s 39ms/step
Archivo guardado en ./datos/submission_LSTM apil

Vemos entonces que estas estrategias de regularización no funcionaron.

## 4 - Embeddings con redes neuronales que usan capas Transformers 

En las secciones 2 y 3, cada modelo aprendió **su propia representación del texto desde cero**: `TextVectorization` le asigna un índice a cada palabra y la capa `Embedding` aprende un vector por palabra usando únicamente nuestras ~18.400 reseñas de entrenamiento. Esa representación tiene dos limitaciones:

1. **Es estática.** La palabra "batería" recibe exactamente el mismo vector en *"la batería dura mucho"* y en *"la batería no dura nada"*. Toda la información de contexto la tiene que reconstruir la red recurrente que viene después.
2. **Solo conoce lo que vio en `train`**, que es ~65 % español. Una palabra en francés o portugués que aparece pocas veces queda con un vector mal entrenado, y una palabra que nunca apareció en `train` se convierte en el token desconocido.

En esta sección probamos **embeddings contextuales**: vectores producidos por un modelo *Transformer encoder-only* preentrenado con enormes cantidades de texto en muchos idiomas. En estos modelos, el vector de cada palabra se calcula mirando **toda la reseña** (mecanismo de atención), así que la misma palabra recibe vectores distintos según el contexto en el que aparece.

Un encoder así se puede usar de dos maneras:

| | ¿Qué se entrena? | Sección |
|---|---|---|
| **Encoder congelado** | Solo la red que va encima. El Transformer funciona como un vectorizador fijo. | 4.1 – 4.11 |
| **Fine-tuning** | El Transformer **y** la red de encima, de punta a punta. | Parte siguiente |

**Pregunta que motiva esta sección:** ¿si dejamos fijas las arquitecturas de las secciones 2 y 3 (MLP, SimpleRNN, LSTM, GRU…) y solo cambiamos la representación de entrada (embedding aprendido desde cero → embeddings contextuales de XLM-R), ¿cambia el desempeño en validación? ¿Cambia igual en todos los idiomas? No suponemos la respuesta: la medimos sobre **la misma partición de validación** de la sección 2.

> **Sobre la aclaración del equipo docente:** solo se permiten arquitecturas *encoder-only* para construir embeddings, con una cabeza de clasificación encima. XLM-RoBERTa es encoder-only (como BERT y RoBERTa) y no genera texto, así que cumple la restricción. Además, a diferencia de BERT y RoBERTa es un modelo entrenado en idiomas distintos al inglés y es la razón rpincipal por la que optamos por este.

### 4.1 - Preparación del entorno

Usamos **PyTorch** con la librería **`transformers`** de Hugging Face para cargar XLM-R y extraer los vectores, porque es la forma estándar de usar estos modelos preentrenados. Los modelos que entrenamos encima de los vectores siguen siendo los mismos modelos de las secciones 2 y 3.

In [39]:
%pip uninstall -y torch

Found existing installation: torch 2.14.1+cu126
Uninstalling torch-2.14.1+cu126:
  Successfully uninstalled torch-2.14.1+cu126
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


In [40]:
%pip install torch --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126
  Using cached torch-2.14.1%2Bcu126-cp311-cp311-win_amd64.whl.metadata (37 kB)
Using cached torch-2.14.1%2Bcu126-cp311-cp311-win_amd64.whl (2602.8 MB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [41]:
import time
import torch
import numpy as np
from transformers import AutoModel, AutoTokenizer

dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo para XLM-R: {dispositivo}")
if dispositivo.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Semillas, para que los resultados sean replicables. set_random_seed fija las de Python,
# NumPy y TensorFlow. Aun así, algunas operaciones en GPU no son 100 % deterministas, así
# que pueden aparecer diferencias pequeñas (décimas de punto) entre ejecuciones.
SEMILLA = 42
tf.keras.utils.set_random_seed(SEMILLA)
torch.manual_seed(SEMILLA)

Dispositivo para XLM-R: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


### 4.2 - El modelo base: XLM-RoBERTa

**XLM-RoBERTa** (Conneau et al., 2020) es la versión multilingüe de RoBERTa:

- Se preentrenó con la tarea de **modelo de lenguaje enmascarado**: se tapan palabras al azar de un texto y el modelo debe adivinarlas a partir del resto. Para hacerlo bien, tiene que aprender cómo se relacionan las palabras entre sí.
- Se entrenó con ~2,5 TB de texto web (CommonCrawl) en **100 idiomas**, entre ellos los cuatro de nuestro problema.
- La versión *base* tiene **12 capas Transformer** que producen vectores de **768 dimensiones** por token (los valores exactos se imprimen abajo).

| Referencia | |
|---|---|
| Modelo | `FacebookAI/xlm-roberta-base` |
| Fuente | https://huggingface.co/FacebookAI/xlm-roberta-base |
| Licencia | MIT |
| Artículo | Conneau et al. (2020). *Unsupervised Cross-lingual Representation Learning at Scale*. https://arxiv.org/abs/1911.02116 |

En esta parte el modelo queda **congelado**: lo ponemos en modo evaluación (`eval()`, que apaga el dropout interno) y marcamos sus parámetros con `requires_grad = False`, de modo que ningún entrenamiento los modifica.

In [42]:
NOMBRE_MODELO_BASE = "FacebookAI/xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(NOMBRE_MODELO_BASE)
modelo_base = AutoModel.from_pretrained(NOMBRE_MODELO_BASE).to(dispositivo)

# Encoder congelado: modo inferencia y sin gradientes para ninguno de sus pesos
modelo_base.eval()
for parametro in modelo_base.parameters():
    parametro.requires_grad = False

DIM = modelo_base.config.hidden_size  # dimensión de cada vector contextual

print(f"Capas Transformer:      {modelo_base.config.num_hidden_layers}")
print(f"Dimensión de los vectores: {DIM}")
print(f"Parámetros:             {sum(p.numel() for p in modelo_base.parameters()) / 1e6:.0f} millones")
print(f"Tamaño del vocabulario: {len(tokenizer):,} subpalabras")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Capas Transformer:      12
Dimensión de los vectores: 768
Parámetros:             278 millones
Tamaño del vocabulario: 250,002 subpalabras


### 4.3 - Tokenización en subpalabras

XLM-R no trabaja con palabras completas, sino con **subpalabras** (algoritmo *SentencePiece*): las palabras frecuentes quedan como un solo token y las raras se parten en pedazos (por ejemplo, `▁resolucion` → `▁resolu` + `cion`; el símbolo `▁` marca el inicio de una palabra). Para nuestros datos esto tiene dos ventajas:

- **No hay palabras desconocidas.** Cualquier palabra (con errores de tipeo, sin tildes, en cualquiera de los cuatro idiomas) se puede escribir con subpalabras del vocabulario.
- **Un solo vocabulario para todos los idiomas.** Pedazos como `▁bater` sirven para "batería" (es), "bateria" (pt) y "batterie" (fr) a la vez.

Además, el tokenizador agrega tokens especiales: `<s>` al inicio (equivale al `[CLS]` de BERT), `</s>` al final y `<pad>` para rellenar las reseñas cortas hasta una longitud fija. La **máscara de atención** (`attention_mask`) vale 1 en los tokens reales y 0 en el relleno, para que el modelo ignore el relleno.

Primero veamos cómo se tokeniza una reseña de cada idioma:

In [43]:
for _, fila in dataset_train.groupby("idioma").head(1).iterrows():
    tokens = tokenizer.tokenize(fila["text"])
    print(f"[{fila['idioma']}] {len(fila['text'].split())} palabras -> {len(tokens)} tokens")
    print(f"     {tokens[:18]} ...\n")

[es] 43 palabras -> 57 tokens
     ['▁Lo', '▁pe', 'dí', '▁un', '▁martes', '▁por', '▁internet', '▁y', '▁llegó', '▁al', '▁tercer', '▁día', ',', '▁viene', '▁en', '▁color', '▁negro', '.'] ...

[fr] 80 palabras -> 117 tokens
     ['▁Le', '▁route', 'ur', '▁cap', 'te', '▁très', '▁bien', '▁dans', '▁toute', '▁la', '▁maison', ',', '▁le', '▁signal', '▁passe', '▁même', '▁à', '▁l'] ...

[en] 57 palabras -> 77 tokens
     ['▁I', '▁got', '▁it', '▁a', '▁while', '▁back', '▁on', '▁sale', '▁online', ',', '▁it', '▁arrived', '▁in', '▁about', '▁four', '▁days', '.', '▁It'] ...

[pt_br] 49 palabras -> 67 tokens
     ['▁Compre', 'i', '▁esse', '▁teclado', '▁mês', '▁passado', '▁pra', '▁trabalhar', '▁em', '▁casa', ',', '▁chegou', '▁em', '▁quatro', '▁dias', '▁numa', '▁caixa', '▁bem'] ...

[es_en] 58 palabras -> 84 tokens
     ['▁Le', '▁regal', 'é', '▁este', '▁parlant', 'e', '▁bluetooth', '▁a', '▁mi', '▁hermano', '▁en', '▁diciembre', '▁y', '▁la', '▁verdad', ',', '▁qué', "▁'"] ...



Para guardar los vectores necesitamos una **longitud fija** por reseña (`MAX_LEN`). En vez de suponerla, la elegimos a partir de la distribución real de longitudes en tokens de `train` y `eval`: tomamos el máximo observado (redondeado hacia arriba a un múltiplo de 8, que es más eficiente en GPU), con un tope de 256 tokens.

In [44]:
textos_todos = list(dataset_train["text"]) + list(dataset_eval["text"])
longitudes = pd.Series([len(ids) for ids in tokenizer(textos_todos)["input_ids"]])  # incluye <s> y </s>

print(longitudes.describe(percentiles=[0.5, 0.9, 0.99]).round(1).to_string())

MAX_LEN = int(min(256, np.ceil(longitudes.max() / 8) * 8))
print(f"\nMAX_LEN = {MAX_LEN} tokens")
print(f"Reseñas que se recortan: {(longitudes > MAX_LEN).sum()} ({(longitudes > MAX_LEN).mean():.2%})")

count    27247.0
mean        80.6
std         20.6
min          7.0
50%         79.0
90%        107.0
99%        129.0
max        176.0

MAX_LEN = 176 tokens
Reseñas que se recortan: 0 (0.00%)


### 4.4 - ¿Qué significa que un embedding sea "contextual"? Una comprobación rápida

Antes de usar los vectores, comprobemos la propiedad que los motiva. Tomamos la palabra **"banco"** en tres oraciones (dos con el sentido de *asiento* y una con el de *entidad financiera*) y calculamos la **similitud coseno** entre los tres vectores que XLM-R le asigna (1 = misma dirección).

- Con el `Embedding` de la sección 3, "banco" tendría **el mismo vector** en las tres oraciones (similitud exactamente 1).
- Si XLM-R produce vectores contextuales, las similitudes deberían ser **menores que 1**, y esperaríamos que los dos usos como *asiento* se parezcan más entre sí que al uso financiero. Lo verificamos en vez de asumirlo.

Como "banco" puede partirse en varias subpalabras, su vector es el **promedio** de los vectores de sus subpalabras.

In [45]:
@torch.no_grad()
def vector_de_palabra(oracion, palabra):
    # Vector contextual de `palabra` dentro de `oracion`: promedio de sus subpalabras en la última capa
    codificacion = tokenizer(oracion, return_offsets_mapping=True, return_tensors="pt")
    posiciones = codificacion.pop("offset_mapping")[0].tolist()  # (inicio, fin) en caracteres de cada token
    estados = modelo_base(**codificacion.to(dispositivo)).last_hidden_state[0].cpu()

    inicio = oracion.lower().index(palabra.lower())
    fin = inicio + len(palabra)
    indices = [i for i, (a, b) in enumerate(posiciones) if b > a and a < fin and b > inicio]
    return estados[indices].mean(dim=0)


oraciones = {
    "asiento 1": "Me senté en el banco del parque a leer un rato.",
    "asiento 2": "El banco de madera del jardín ya está podrido.",
    "financiero": "Fui al banco a pedir un préstamo para el carro.",
}
vectores = {nombre: vector_de_palabra(o, "banco") for nombre, o in oraciones.items()}
nombres = list(vectores)

similitudes = pd.DataFrame(
    [[torch.cosine_similarity(vectores[a], vectores[b], dim=0).item() for b in nombres] for a in nombres],
    index=nombres,
    columns=nombres,
)
similitudes.round(3)

,asiento 1,asiento 2,financiero
asiento 1,1.000,0.997,0.992
asiento 2,0.997,1.000,0.993
financiero,0.992,0.993,1.000


**Lectura del resultado:** *(completar después de ejecutar)*. ¿Las similitudes son menores que 1? ¿"asiento 1" se parece más a "asiento 2" que a "financiero"? Ojo: en los modelos Transformer todas las similitudes coseno suelen ser altas (los vectores ocupan una región angosta del espacio), así que lo que importa es la **comparación relativa**, no el valor absoluto.

### 4.5 - Extracción de los embeddings (encoder congelado)

Pasamos todas las reseñas por XLM-R **una sola vez** (solo inferencia, sin gradientes) y guardamos sus salidas para entrenar después todos los modelos Keras sin volver a ejecutar el Transformer.

Para cada lote, XLM-R devuelve:
- `last_hidden_state`: los vectores de la **última capa**, con forma `(reseñas, MAX_LEN, 768)`: un vector contextual por token.
- `hidden_states`: las salidas de **todas** las capas (13: la de embeddings más las 12 capas Transformer).

A partir de ellas guardamos dos tipos de representación:

| Representación | Forma por reseña | Para qué modelo |
|---|---|---|
| **Secuencia de vectores** de la última capa (con ceros en el relleno) | `(MAX_LEN, 768)` | Redes recurrentes (sección 4.8): reemplaza a `TextVectorization` + `Embedding` |
| **Un solo vector** por reseña (*pooling*) | `(768,)` | MLP (sección 4.7): reemplaza al vector TF-IDF |

Para obtener un solo vector por reseña hay varias opciones razonables, y no sabemos de antemano cuál funciona mejor aquí, así que guardamos **tres** y las comparamos:
1. **`cls`**: el vector del token `<s>` de la última capa (en BERT, este token se usa para clasificar).
2. **`media`**: el promedio de los vectores de todos los tokens reales de la última capa (sin contar el relleno).
3. **`media_4_capas`**: igual que `media`, pero promediando además las 4 últimas capas. Las capas intermedias guardan información más "léxica" y la última más "contextual", así que combinarlas podría ayudar.

Detalles de implementación:
- `torch.no_grad()` desactiva el cálculo de gradientes: ahorra memoria y tiempo porque no vamos a entrenar el encoder.
- En GPU usamos `torch.autocast` con `float16` (precisión media), que es ~2 veces más rápido. Las secuencias se **guardan** en `float16` para ocupar la mitad de memoria; la pérdida de precisión es despreciable para estos vectores.
- Ponemos en **cero** los vectores de las posiciones de relleno. Así, la capa `Masking` de Keras (sección 4.8) puede reconocer el relleno y la red recurrente lo ignora, igual que hacía `mask_zero=True` en la sección 3.

In [46]:
@torch.no_grad()
def extraer_embeddings(textos, tamano_lote=64):
    # Pasa `textos` por XLM-R congelado y devuelve:
    #   - secuencias: array (n, MAX_LEN, DIM) en float16, con ceros en las posiciones de relleno
    #   - vectores: dict con un array (n, DIM) por cada tipo de pooling ('cls', 'media', 'media_4_capas')
    # Usa padding dinámico: ordena las reseñas por longitud y rellena cada lote solo hasta
    # la reseña más larga de ese lote, en vez de hasta MAX_LEN. Los resultados quedan en el
    # orden original de `textos`.
    n = len(textos)
    secuencias = np.zeros((n, MAX_LEN, DIM), dtype=np.float16)
    vectores = {tipo: np.zeros((n, DIM), dtype=np.float32) for tipo in ["cls", "media", "media_4_capas"]}
    usar_fp16 = dispositivo.type == "cuda"

    # Orden de las reseñas de la más corta a la más larga (en tokens)
    longitudes = [len(ids) for ids in tokenizer(textos, truncation=True, max_length=MAX_LEN)["input_ids"]]
    orden = np.argsort(longitudes)

    for numero_lote, inicio in enumerate(range(0, n, tamano_lote)):
        posiciones = orden[inicio : inicio + tamano_lote]  # posiciones originales de las reseñas de este lote
        lote = [textos[i] for i in posiciones]
        codificacion = tokenizer(
            lote, padding="longest", truncation=True, max_length=MAX_LEN, return_tensors="pt"
        ).to(dispositivo)
        largo_lote = codificacion["input_ids"].shape[1]  # longitud de la reseña más larga del lote

        with torch.autocast(device_type=dispositivo.type, dtype=torch.float16, enabled=usar_fp16):
            salida = modelo_base(**codificacion, output_hidden_states=True)

        mascara = codificacion["attention_mask"].unsqueeze(-1).float()  # (lote, largo_lote, 1): 1 = token real
        n_tokens = mascara.sum(dim=1)  # (lote, 1): cuántos tokens reales tiene cada reseña

        ultima_capa = salida.last_hidden_state.float() * mascara  # pone en cero el relleno
        cuatro_capas = torch.stack(salida.hidden_states[-4:]).float().mean(dim=0) * mascara

        # Se guarda en las posiciones originales; de largo_lote a MAX_LEN el array ya tiene ceros
        secuencias[posiciones, :largo_lote] = ultima_capa.cpu().numpy().astype(np.float16)
        vectores["cls"][posiciones] = ultima_capa[:, 0].cpu().numpy()  # posición 0 = token <s>
        vectores["media"][posiciones] = (ultima_capa.sum(dim=1) / n_tokens).cpu().numpy()
        vectores["media_4_capas"][posiciones] = (cuatro_capas.sum(dim=1) / n_tokens).cpu().numpy()

        if numero_lote % 50 == 0:
            print(f"  {inicio + len(lote):>6,} / {n:,} reseñas")

    return secuencias, vectores

In [47]:
t0 = time.time()
print("Extrayendo embeddings de train...")
secuencias_train, vectores_train = extraer_embeddings(list(dataset_train["text"]))
print("Extrayendo embeddings de eval...")
secuencias_eval, vectores_eval = extraer_embeddings(list(dataset_eval["text"]))
print(f"\nTiempo total de extracción: {(time.time() - t0) / 60:.1f} minutos")

print(f"Secuencias train: {secuencias_train.shape}  ({secuencias_train.nbytes / 1e9:.2f} GB en memoria)")
print(f"Secuencias eval:  {secuencias_eval.shape}  ({secuencias_eval.nbytes / 1e9:.2f} GB en memoria)")
print(f"Vectores por reseña (cada pooling): {vectores_train['cls'].shape}")

# Los vectores por reseña ocupan poco, así que los guardamos en disco para no tener que
# volver a extraerlos si se reinicia la sesión. Las secuencias (varios GB) no se guardan.
np.savez_compressed(
    "./datos/vectores_xlmr_congelado.npz",
    **{f"train_{k}": v for k, v in vectores_train.items()},
    **{f"eval_{k}": v for k, v in vectores_eval.items()},
)

# Liberamos la memoria de GPU que ocupa XLM-R para dejársela a TensorFlow
del modelo_base
if dispositivo.type == "cuda":
    torch.cuda.empty_cache()

Extrayendo embeddings de train...
      64 / 20,466 reseñas
   3,264 / 20,466 reseñas
   6,464 / 20,466 reseñas
   9,664 / 20,466 reseñas
  12,864 / 20,466 reseñas
  16,064 / 20,466 reseñas
  19,264 / 20,466 reseñas
Extrayendo embeddings de eval...
      64 / 6,781 reseñas
   3,264 / 6,781 reseñas
   6,464 / 6,781 reseñas

Tiempo total de extracción: 1.2 minutos
Secuencias train: (20466, 176, 768)  (5.53 GB en memoria)
Secuencias eval:  (6781, 176, 768)  (1.83 GB en memoria)
Vectores por reseña (cada pooling): (20466, 768)


### 4.6 - Misma partición de entrenamiento y validación que en la sección 2

Para que la comparación sea justa, los modelos de esta sección se entrenan y validan con **exactamente las mismas reseñas** que los de las secciones 2 y 3. `train_df` y `test_df` conservan el índice original de `dataset_train`, así que basta con indexar los arrays de vectores con esos índices.

In [48]:
indices_train = train_df.index.to_numpy()
indices_val = test_df.index.to_numpy()

# Etiquetas de todo dataset_train como enteros; cada lote toma las de sus índices
y_todas = dataset_train["label"].map(etiqueta_a_id).to_numpy()

print(f"Entrenamiento: {len(indices_train):,} reseñas | Validación: {len(indices_val):,} reseñas")

Entrenamiento: 18,419 reseñas | Validación: 2,047 reseñas


**¿Cómo le pasamos varios GB de vectores a Keras?** En la sección 2 los datos se entregaban como `tf.data.Dataset` creados con `from_tensor_slices`, y `model.fit` también acepta arrays de NumPy. Pero ambas opciones convierten el array completo en un tensor de TensorFlow, es decir, hacen **otra copia completa en memoria**. Con las secuencias (varios GB) eso puede agotar la RAM.

Por eso definimos `LotesNumpy`, una subclase de `tf.keras.utils.PyDataset`: un objeto que Keras recorre **lote por lote**. Cada vez que Keras pide el lote `i`, se toman de NumPy solo las 32 reseñas de ese lote (según sus índices), se convierten a `float32` y se entregan. Así nunca se copia el array completo. Al final de cada época se barajan los índices, igual que hacía `shuffle` en la sección 2.

La función de entrenamiento es equivalente a la de la sección 2: misma parada temprana sobre `val_accuracy`, misma paciencia de 3 épocas y restauración de los mejores pesos.

In [49]:
class LotesNumpy(tf.keras.utils.PyDataset):
    # Entrega a Keras, lote por lote, las filas `indices` del array `X` (y sus etiquetas, si se dan)

    def __init__(self, X, indices, y=None, tamano_lote=32, barajar=False, **kwargs):
        super().__init__(**kwargs)
        self.X, self.y = X, y
        self.indices = np.array(indices)
        self.tamano_lote, self.barajar = tamano_lote, barajar
        self.on_epoch_end()

    def __len__(self):  # número de lotes por época
        return int(np.ceil(len(self.indices) / self.tamano_lote))

    def __getitem__(self, i):
        filas = self.indices[i * self.tamano_lote : (i + 1) * self.tamano_lote]
        X_lote = self.X[filas].astype(np.float32)
        return X_lote if self.y is None else (X_lote, self.y[filas])

    def on_epoch_end(self):
        if self.barajar:
            np.random.shuffle(self.indices)


def entrenar_con_parada_temprana_np(model, X, max_epochs=50, patience=3, tamano_lote=32):
    # Igual que entrenar_modelo_con_parada_temprana (sección 2), pero los datos salen del array X
    # (vectores o secuencias de XLM-R de todo dataset_train) usando los índices de train_df y test_df
    datos_train = LotesNumpy(X, indices_train, y_todas, tamano_lote, barajar=True)
    datos_val = LotesNumpy(X, indices_val, y_todas, tamano_lote)
    parada_temprana = tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", mode="max", patience=patience, restore_best_weights=True
    )
    history = model.fit(
        datos_train,
        validation_data=datos_val,
        epochs=max_epochs,
        callbacks=[parada_temprana],
        verbose=2,  # una línea por época
    )
    mejor_epoca = int(np.argmax(history.history["val_accuracy"]))
    print(f"Mejor época: {mejor_epoca + 1} (val_accuracy = {history.history['val_accuracy'][mejor_epoca]:.4f})")
    return history


def predecir(model, X, indices):
    # Clase predicha (0, 1 o 2) para las filas `indices` del array X
    return model.predict(LotesNumpy(X, indices, tamano_lote=64), verbose=0).argmax(axis=1)


# Aquí guardamos los modelos e historiales de esta sección, para compararlos al final
historiales_contextuales = {}
modelos_contextuales = {}  # nombre -> (modelo, X de train, X de eval)

### 4.7 - MLP sobre un vector contextual por reseña

Es **la misma arquitectura del MLP de la sección 2** (`Dense(32, relu)` → `Dense(3)`); lo único que cambia es la entrada:

| | Sección 2 | Sección 4.7 |
|---|---|---|
| Entrada | Vector TF-IDF de ~19.000 dimensiones (una por palabra del vocabulario, sin orden ni contexto) | Vector de 768 dimensiones producido por XLM-R |
| Parámetros de la primera capa | ~609.000 | 768 × 32 + 32 = 24.608 |

Entrenamos un MLP por cada tipo de pooling (`cls`, `media`, `media_4_capas`) para ver cuál conviene.

In [50]:
def construir_mlp_contextual(nombre):
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(DIM,)),
            tf.keras.layers.Dense(32, activation="relu"),
            tf.keras.layers.Dense(3),
        ],
        name=nombre,
    )
    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=tf.keras.optimizers.Adam(),
        metrics=["accuracy"],
    )
    return model


for tipo in ["cls", "media", "media_4_capas"]:
    nombre = f"0 - MLP [{tipo}]"
    print(f"\n===== {nombre} =====")
    modelo = construir_mlp_contextual(nombre=f"mlp_{tipo}")
    historiales_contextuales[nombre] = entrenar_con_parada_temprana_np(modelo, vectores_train[tipo])
    modelos_contextuales[nombre] = (modelo, vectores_train[tipo], vectores_eval[tipo])


===== 0 - MLP [cls] =====
Epoch 1/50
576/576 - 7s - 13ms/step - accuracy: 0.5249 - loss: 0.9400 - val_accuracy: 0.6106 - val_loss: 0.7703
Epoch 2/50
576/576 - 2s - 3ms/step - accuracy: 0.6461 - loss: 0.6966 - val_accuracy: 0.6742 - val_loss: 0.6583
Epoch 3/50
576/576 - 2s - 3ms/step - accuracy: 0.6649 - loss: 0.6299 - val_accuracy: 0.6629 - val_loss: 0.6159
Epoch 4/50
576/576 - 1s - 2ms/step - accuracy: 0.6771 - loss: 0.5991 - val_accuracy: 0.6844 - val_loss: 0.5914
Epoch 5/50
576/576 - 2s - 3ms/step - accuracy: 0.6848 - loss: 0.5828 - val_accuracy: 0.6644 - val_loss: 0.5925
Epoch 6/50
576/576 - 2s - 4ms/step - accuracy: 0.6954 - loss: 0.5646 - val_accuracy: 0.6590 - val_loss: 0.5769
Epoch 7/50
576/576 - 2s - 3ms/step - accuracy: 0.7010 - loss: 0.5542 - val_accuracy: 0.6600 - val_loss: 0.5709
Mejor época: 4 (val_accuracy = 0.6844)

===== 0 - MLP [media] =====
Epoch 1/50
576/576 - 2s - 4ms/step - accuracy: 0.5898 - loss: 0.8156 - val_accuracy: 0.6634 - val_loss: 0.6480
Epoch 2/50
576/5

### 4.8 - Redes recurrentes sobre la secuencia de vectores contextuales

Repetimos **las mismas cinco configuraciones de la sección 3**, con la misma cabeza (`Dense(32, relu)` → `Dense(3)`). Lo único que cambia es el inicio de la red:

```
Sección 3:   texto → TextVectorization → Embedding (aprendido desde cero, 64 dim) → capas recurrentes → Dense(32) → Dense(3)
Sección 4.8: texto → XLM-R congelado (768 dim por token, calculado en 4.5)     → capas recurrentes → Dense(32) → Dense(3)
```

La capa `Masking(mask_value=0.0)` marca como relleno los pasos de tiempo cuyo vector es todo ceros (los que pusimos en cero en 4.5), para que las capas recurrentes los salten. Cumple el mismo papel que `mask_zero=True` en la sección 3.

Las configuraciones se definen como funciones (`lambda`) para crear **capas nuevas** en cada modelo: una capa de Keras no se puede reutilizar en dos modelos distintos sin compartir sus pesos.

In [51]:
def construir_modelo_rnn_contextual(capas_recurrentes, nombre):
    # Igual que construir_modelo_rnn (sección 3), pero la entrada es la secuencia de vectores de XLM-R
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(MAX_LEN, DIM)),
            tf.keras.layers.Masking(mask_value=0.0),
            *capas_recurrentes,
            tf.keras.layers.Dense(32, activation="relu"),
            tf.keras.layers.Dense(3),
        ],
        name=nombre,
    )
    model.compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=tf.keras.optimizers.Adam(),
        metrics=["accuracy"],
    )
    return model


# Las mismas cinco configuraciones de la sección 3, con los mismos nombres para poder compararlas
configuraciones_rnn = {
    "1 - SimpleRNN bidireccional": lambda: [tf.keras.layers.Bidirectional(tf.keras.layers.SimpleRNN(64))],
    "2 - SimpleRNN apiladas": lambda: [
        tf.keras.layers.SimpleRNN(64, return_sequences=True),
        tf.keras.layers.SimpleRNN(32),
    ],
    "3 - LSTM": lambda: [tf.keras.layers.LSTM(64)],
    "4 - GRU bidireccional + dropout": lambda: [
        tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64)),
        tf.keras.layers.Dropout(0.5),
    ],
    "5 - LSTM bidireccionales apiladas": lambda: [
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
        tf.keras.layers.Dropout(0.5),
    ],
}

In [52]:
for nombre, crear_capas in configuraciones_rnn.items():
    print(f"\n===== {nombre} (XLM-R congelado) =====")
    nombre_keras = "xlmr_" + nombre.split(" - ")[1].lower().replace(" ", "_").replace("+", "mas")
    modelo = construir_modelo_rnn_contextual(crear_capas(), nombre=nombre_keras)
    t0 = time.time()
    historiales_contextuales[nombre] = entrenar_con_parada_temprana_np(modelo, secuencias_train)
    print(f"Tiempo de entrenamiento: {(time.time() - t0) / 60:.1f} minutos")
    modelos_contextuales[nombre] = (modelo, secuencias_train, secuencias_eval)


===== 1 - SimpleRNN bidireccional (XLM-R congelado) =====
Epoch 1/50
576/576 - 149s - 259ms/step - accuracy: 0.5725 - loss: 0.7293 - val_accuracy: 0.6175 - val_loss: 0.5991
Epoch 2/50
576/576 - 132s - 230ms/step - accuracy: 0.6200 - loss: 0.6744 - val_accuracy: 0.6805 - val_loss: 0.5873
Epoch 3/50
576/576 - 124s - 215ms/step - accuracy: 0.6800 - loss: 0.5911 - val_accuracy: 0.6888 - val_loss: 0.6279
Epoch 4/50
576/576 - 119s - 207ms/step - accuracy: 0.7613 - loss: 0.5083 - val_accuracy: 0.7973 - val_loss: 0.4350
Epoch 5/50
576/576 - 122s - 213ms/step - accuracy: 0.7886 - loss: 0.4730 - val_accuracy: 0.7851 - val_loss: 0.4782
Epoch 6/50
576/576 - 120s - 209ms/step - accuracy: 0.8044 - loss: 0.4326 - val_accuracy: 0.7841 - val_loss: 0.4478
Epoch 7/50
576/576 - 110s - 192ms/step - accuracy: 0.8232 - loss: 0.3908 - val_accuracy: 0.8056 - val_loss: 0.4007
Epoch 8/50
576/576 - 115s - 200ms/step - accuracy: 0.8256 - loss: 0.3721 - val_accuracy: 0.7846 - val_loss: 0.4483
Epoch 9/50
576/576 - 

### 4.9 - Comparación: embedding aprendido desde cero vs. XLM-R congelado

Juntamos en una tabla los resultados de las secciones 2-3 y de esta sección. Para cada modelo reportamos el `val_accuracy` de su mejor época y el `train_accuracy` de esa misma época. La diferencia entre los dos es una medida del **sobreajuste**.

In [53]:
def resumir_historiales(historiales_dict, representacion):
    filas = []
    for nombre, h in historiales_dict.items():
        mejor = int(np.argmax(h.history["val_accuracy"]))
        filas.append(
            {
                "modelo": nombre,
                "representación": representacion,
                "mejor_época": mejor + 1,
                "train_accuracy": h.history["accuracy"][mejor],
                "val_accuracy": h.history["val_accuracy"][mejor],
            }
        )
    return filas


# Resultados previos: MLP con TF-IDF (sección 2) y las cinco redes recurrentes (sección 3)
historiales_previos = dict(historiales)
if "historial_mlp" in globals():
    historiales_previos = {"0 - MLP [tf-idf]": historial_mlp, **historiales_previos}

comparacion = pd.DataFrame(
    resumir_historiales(historiales_previos, "Desde cero (secc. 2-3)")
    + resumir_historiales(historiales_contextuales, "XLM-R congelado (secc. 4)")
)
comparacion["sobreajuste"] = comparacion["train_accuracy"] - comparacion["val_accuracy"]
comparacion.sort_values("val_accuracy", ascending=False).round(4)

,modelo,representación,mejor_época,train_accuracy,val_accuracy,sobreajuste
12,4 - GRU bidireccional + dropout,XLM-R congelado (secc. 4),12,0.9282,0.8925,0.0356
13,5 - LSTM bidireccionales apiladas,XLM-R congelado (secc. 4),11,0.9014,0.8852,0.0162
11,3 - LSTM,XLM-R congelado (secc. 4),6,0.8898,0.8832,0.0066
10,2 - SimpleRNN apiladas,XLM-R congelado (secc. 4),8,0.8455,0.8388,0.0068
9,1 - SimpleRNN bidireccional,XLM-R congelado (secc. 4),11,0.8519,0.8383,0.0136
4,4 - GRU bidireccional + dropout,Desde cero (secc. 2-3),4,0.9188,0.8168,0.1020
3,3 - LSTM,Desde cero (secc. 2-3),4,0.9022,0.8119,0.0903
5,5 - LSTM bidireccionales apiladas,Desde cero (secc. 2-3),6,0.9282,0.8061,0.1221
8,0 - MLP [media_4_capas],XLM-R congelado (secc. 4),17,0.8113,0.7982,0.0131
1,1 - SimpleRNN bidireccional,Desde cero (secc. 2-3),2,0.8192,0.7753,0.0439


In [54]:
# Vista lado a lado de las redes recurrentes: misma arquitectura, distinta representación de entrada
tabla_rnn = comparacion[comparacion["modelo"].isin(configuraciones_rnn)].pivot(
    index="modelo", columns="representación", values="val_accuracy"
)
tabla_rnn["diferencia"] = tabla_rnn["XLM-R congelado (secc. 4)"] - tabla_rnn["Desde cero (secc. 2-3)"]
tabla_rnn.round(4)

representación,Desde cero (secc. 2-3),XLM-R congelado (secc. 4),diferencia
modelo,,,
1 - SimpleRNN bidireccional,0.7753,0.8383,0.0630
2 - SimpleRNN apiladas,0.7553,0.8388,0.0835
3 - LSTM,0.8119,0.8832,0.0713
4 - GRU bidireccional + dropout,0.8168,0.8925,0.0757
5 - LSTM bidireccionales apiladas,0.8061,0.8852,0.0791


### 4.10 - Análisis de errores por idioma (partición de validación)

El enunciado advierte que la distribución de idiomas de `eval` no es la misma que la de `train`, así que no basta con el accuracy global: nos interesa saber **en qué idiomas falla cada modelo**. Comparamos el mejor modelo de la sección 3 con el mejor de esta sección, idioma por idioma, sobre las mismas reseñas de validación.

Hay que tener en cuenta el tamaño de cada grupo: con ~30 reseñas bilingües (`es_en`) en validación, una diferencia de 1 o 2 aciertos ya cambia el accuracy en varios puntos, así que esa fila es poco confiable.

In [55]:
# Mejor modelo de la sección 3 (entrada: texto) y mejor de la sección 4 (entrada: vectores de XLM-R)
modelos_seccion3 = dict(zip(historiales.keys(), [model_rnn_1, model_rnn_2, model_rnn_3, model_rnn_4, model_rnn_5]))
resumen_previo = comparacion[comparacion["modelo"].isin(modelos_seccion3)
                             & (comparacion["representación"] == "Desde cero (secc. 2-3)")]
nombre_mejor_s3 = resumen_previo.sort_values("val_accuracy").iloc[-1]["modelo"]
resumen_s4 = comparacion[comparacion["representación"] == "XLM-R congelado (secc. 4)"]
nombre_mejor_s4 = resumen_s4.sort_values("val_accuracy").iloc[-1]["modelo"]
print(f"Mejor de la sección 3: {nombre_mejor_s3}")
print(f"Mejor de la sección 4: {nombre_mejor_s4}")

# Predicciones de cada uno sobre las mismas reseñas de validación
textos_val = tf.data.Dataset.from_tensor_slices(test_df["text"].values).batch(64)
pred_s3 = modelos_seccion3[nombre_mejor_s3].predict(textos_val, verbose=0).argmax(axis=1)
modelo_s4, X_train_s4, _ = modelos_contextuales[nombre_mejor_s4]
pred_s4 = predecir(modelo_s4, X_train_s4, indices_val)

analisis_val = test_df[["text", "idioma", "label"]].copy()
analisis_val["pred_s3"] = [etiquetas[p] for p in pred_s3]
analisis_val["pred_s4"] = [etiquetas[p] for p in pred_s4]
analisis_val["acierto_s3"] = analisis_val["pred_s3"] == analisis_val["label"]
analisis_val["acierto_s4"] = analisis_val["pred_s4"] == analisis_val["label"]

por_idioma = analisis_val.groupby("idioma").agg(
    n_reseñas=("label", "size"),
    acc_seccion_3=("acierto_s3", "mean"),
    acc_seccion_4=("acierto_s4", "mean"),
)
por_idioma.loc["TOTAL"] = [len(analisis_val), analisis_val["acierto_s3"].mean(), analisis_val["acierto_s4"].mean()]
por_idioma["diferencia"] = por_idioma["acc_seccion_4"] - por_idioma["acc_seccion_3"]
por_idioma.round(4)

Mejor de la sección 3: 4 - GRU bidireccional + dropout
Mejor de la sección 4: 4 - GRU bidireccional + dropout


,n_reseñas,acc_seccion_3,acc_seccion_4,diferencia
idioma,,,,
en,234.0,0.7179,0.8675,0.1496
es,1337.0,0.8527,0.9020,0.0494
es_en,33.0,0.6667,0.8182,0.1515
fr,210.0,0.7000,0.8667,0.1667
pt_br,233.0,0.7382,0.8970,0.1588
TOTAL,2047.0,0.8056,0.8925,0.0870


Las matrices de confusión muestran **qué clases** se confunden. Filas: etiqueta real; columnas: etiqueta predicha.

In [56]:
for columna, titulo in [("pred_s3", f"Sección 3 - {nombre_mejor_s3}"), ("pred_s4", f"Sección 4 - {nombre_mejor_s4}")]:
    print(f"\n{titulo}")
    print(pd.crosstab(analisis_val["label"], analisis_val[columna], rownames=["real"], colnames=["predicho"]))


Sección 3 - 4 - GRU bidireccional + dropout
predicho  negativo  neutral  positivo
real                                 
negativo       533       11       194
neutral          1      558         1
positivo       183        8       558

Sección 4 - 4 - GRU bidireccional + dropout
predicho  negativo  neutral  positivo
real                                 
negativo       637        0       101
neutral          0      560         0
positivo       119        0       630


### 4.11 - Archivo de envío y guardado del mejor modelo de esta sección

Generamos el archivo de predicciones de `eval` con el mejor modelo de esta sección (según `val_accuracy`) y guardamos el modelo con `model.save`, como piden los entregables.

In [57]:
def crear_archivo_predicciones_vectores(model, X_eval, df, archivo_salida):
    # Como crear_archivo_predicciones, pero la entrada son los vectores de XLM-R ya calculados
    clases = predecir(model, X_eval, np.arange(len(X_eval)))
    submission = pd.DataFrame({"id": df["id"].values, "answer": [etiquetas[c] for c in clases]})
    submission.to_csv(archivo_salida, index=False)
    print(f"Archivo guardado en {archivo_salida} ({len(submission)} predicciones)")
    print(submission["answer"].value_counts(normalize=True).round(3))
    return submission


modelo_s4, _, X_eval_s4 = modelos_contextuales[nombre_mejor_s4]
nombre_archivo = nombre_mejor_s4.split(" - ")[1].replace(" ", "_").replace("[", "").replace("]", "")
submission_s4 = crear_archivo_predicciones_vectores(
    modelo_s4, X_eval_s4, dataset_eval, f"./datos/submission_xlmr_congelado_{nombre_archivo}.csv"
)
modelo_s4.save(f"./datos/modelo_xlmr_congelado_{nombre_archivo}.keras")

Archivo guardado en ./datos/submission_xlmr_congelado_GRU_bidireccional_+_dropout.csv (6781 predicciones)
answer
negativo    0.413
positivo    0.396
neutral     0.190
Name: proportion, dtype: float64
